# Deploy Pro Matches Model

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from sklearn.linear_model import LogisticRegression

from dota_oracle_common.models.features.schema import TeamFeaturesHyperparams, PlayerHeroFeaturesHyperparams, HeroFeaturesHyperparams
from dota_oracle_common.postgresql import DatabaseManager

from dota_oracle_pipeline.feature_engineering.batch.hero_wr_features.decay import HeroWinrateDecayFeatureGenerator
from dota_oracle_pipeline.feature_engineering.batch.player_hero_features.dynamic_prior import PlayerHeroDynamicPriorFeatureGenerator
from dota_oracle_pipeline.feature_engineering.batch.team_features.decay import TeamDecayFeatureGenerator

from dota_oracle_pipeline.feature_transformation.aggregate_features import create_final_features


In [3]:
TEAM_FEATURES_HYPERPARAMS = TeamFeaturesHyperparams(
    prior_mean=0.52, prior_count=13, half_life_days=45
)

HERO_FEATURES_HYPERPARAMS = HeroFeaturesHyperparams(
    prior_mean=0.5, prior_count=50, half_life_days=45
)

PLAYER_HERO_FEATURES_HYPERPARAMS = PlayerHeroFeaturesHyperparams(
    player_prior_count=8,
    player_half_life_days=60, 
    hero_prior_count=50,
    hero_prior_mean=0.5,
    hero_half_life_days=45
)

In [4]:
local_session = DatabaseManager.get_session_factory()

2025-11-04 04:13:33,742 - dota_oracle_common.postgresql - INFO - Creating database engine with pool_size=10 and max_overflow=5


2025-11-04 04:13:33,780 - dota_oracle_common.postgresql - INFO - Successfully initialized database for 'dota2' at localhost


In [5]:
from dota_oracle_schedules.ml_pipelines.backfill_feature_engineering import get_all_matches, filter_completed_matches


In [6]:
async with local_session() as db_session:
    # Memory bottleneck to consider when number of matches is very large. Change batch processing to stateful generators later. 
    all_historical_matches = await get_all_matches(db_session)
    sorted_matches = sorted(all_historical_matches, key=lambda x: x.start_time)
    completed_matches = filter_completed_matches(sorted_matches)
    
completed_matches[:100]

2025-11-04 04:13:37,991 - dota_oracle_schedules.ml_pipelines.backfill_feature_engineering - INFO - Fetching all match details from the database...
2025-11-04 04:13:49,541 - dota_oracle_common.repositories.base_repository - INFO - Retrieved 127694 records for MatchTable
2025-11-04 04:13:49,542 - dota_oracle_common.repositories.match_repository - INFO - Found 127694 MatchTable details.
2025-11-04 04:13:49,543 - dota_oracle_schedules.ml_pipelines.backfill_feature_engineering - INFO - Fetched 127694 matches from the database.


[MatchTable(slot_129_hero_id=54, slot_0_account_id=452400903, slot_131_account_id=194521913, slot_0_hero_id=42, slot_130_hero_id=7, slot_132_account_id=278770268, slot_131_hero_id=98, slot_2_account_id=493967542, radiant_team_id=8254112, slot_1_hero_id=30, slot_132_hero_id=121, slot_3_account_id=455509466, dire_team_id=8252786, slot_2_hero_id=107, match_id=5999176266, slot_4_account_id=413339999, start_time=datetime.datetime(2021, 5, 17, 21, 6, 51, tzinfo=datetime.timezone.utc), slot_3_hero_id=52, slot_128_account_id=116293223, radiant_name='Omega Gaming', dire_name='Incubus Gaming', slot_4_hero_id=55, slot_129_account_id=173971950, duration=2398.0, slot_128_hero_id=65, leagueid=None, slot_1_account_id=294218576, slot_130_account_id=101779337),
 MatchTable(slot_129_hero_id=17, slot_0_account_id=86818655, slot_131_account_id=184131721, slot_0_hero_id=68, slot_130_hero_id=61, slot_132_account_id=107579895, slot_131_hero_id=88, slot_2_account_id=126174633, radiant_team_id=8375259, slot_1_

In [7]:
train_percentage = 0.8
train_size = int(len(completed_matches) * train_percentage)
train_matches = completed_matches[:train_size]
test_matches = completed_matches[train_size:]
print(f"Train matches: {len(train_matches)}, Test matches: {len(test_matches)}")

train_ids_set = set([match.match_id for match in train_matches])
test_ids_set = set([match.match_id for match in test_matches])


Train matches: 101324, Test matches: 25331


In [8]:
hero_generator = HeroWinrateDecayFeatureGenerator()
hero_features = hero_generator.generate_wide_format(
    completed_matches,
    prior_mean=HERO_FEATURES_HYPERPARAMS.prior_mean,
    prior_count=HERO_FEATURES_HYPERPARAMS.prior_count,
    half_life_days=HERO_FEATURES_HYPERPARAMS.half_life_days,
)


player_hero_generator = PlayerHeroDynamicPriorFeatureGenerator()
player_hero_features = player_hero_generator.generate(
    completed_matches,
    player_prior_count=PLAYER_HERO_FEATURES_HYPERPARAMS.player_prior_count,
    player_half_life_days=PLAYER_HERO_FEATURES_HYPERPARAMS.player_half_life_days,
    hero_prior_count=PLAYER_HERO_FEATURES_HYPERPARAMS.hero_prior_count,
    hero_prior_mean=PLAYER_HERO_FEATURES_HYPERPARAMS.hero_prior_mean,
    hero_half_life_days=PLAYER_HERO_FEATURES_HYPERPARAMS.hero_half_life_days,
)
        
team_generator = TeamDecayFeatureGenerator()
team_features = team_generator.generate(
    completed_matches,
    prior_mean=TEAM_FEATURES_HYPERPARAMS.prior_mean,
    prior_count=TEAM_FEATURES_HYPERPARAMS.prior_count,
    half_life_days=TEAM_FEATURES_HYPERPARAMS.half_life_days,
)

Step 1/2: Generating dynamic hero priors...
Hero priors generated.
Step 2/2: Generating player-hero features with dynamic priors...
Player-hero features generated.


In [9]:
import pandas as pd

In [10]:
aggregated_features = create_final_features(
    hero_features_list=hero_features,
    player_hero_features_list=player_hero_features,
    team_features_list=team_features,
)


['match_id', 'radiant_dire_team_wr_diff', 'radiant_dire_matchup', 'radiant_dire_player_wr_diff', 'radiant_dire_hero_wr_diff']


In [11]:
final_features_df = pd.DataFrame([feat.model_dump() for feat in aggregated_features])
final_features_df

,match_id,radiant_dire_team_wr_diff,radiant_dire_matchup,radiant_dire_player_wr_diff,radiant_dire_hero_wr_diff
0,5999176266,0.000000,0.520000,0.000000,0.000000
1,5999201501,0.000000,0.520000,0.000000,0.000000
2,5999214195,0.000000,0.520000,-0.003919,-0.003919
3,5999249937,0.071386,0.517121,0.019819,-0.001959
4,5999283181,0.000043,0.517357,-0.001880,-0.001884
...,...,...,...,...,...
126650,8541987656,-0.056029,0.520000,-0.013221,0.000608
126651,8542008481,0.000000,0.520000,0.019723,0.019723
126652,8542042026,0.076770,0.520000,0.059289,-0.012409
126653,8542050104,-0.118197,0.482875,0.020261,0.003676


In [12]:
match_outcomes = [match.outcome for match in completed_matches if match.outcome is not None]
match_outcomes_df = pd.DataFrame([match.model_dump() for match in match_outcomes])

match_outcomes_df

,radiant_win,match_id
0,True,5999176266
1,True,5999201501
2,False,5999214195
3,False,5999249937
4,False,5999283181
...,...,...
126650,False,8541987656
126651,True,8542008481
126652,True,8542042026
126653,True,8542050104


In [13]:
final_df = pd.merge(final_features_df, match_outcomes_df, on="match_id")

train_df = final_df[final_df['match_id'].isin(train_ids_set)]
test_df = final_df[final_df['match_id'].isin(test_ids_set)]

len(train_df), len(test_df)

(101324, 25331)

In [14]:
X_train = train_df.drop(columns=["match_id", "radiant_win"])
y_train = train_df["radiant_win"]
X_test = test_df.drop(columns=["match_id", "radiant_win"])
y_test = test_df["radiant_win"]

len(X_train), len(X_test), len(y_train), len(y_test)

(101324, 25331, 101324, 25331)

In [15]:
log_model = LogisticRegression(random_state=42, max_iter=1000)

In [21]:
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss


In [20]:
log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)
y_predict_proba = log_model.predict_proba(X_test)[:, 1]

In [22]:
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_predict_proba)
log_loss_value = log_loss(y_test, y_predict_proba)

print(f"Accuracy: {accuracy:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
print(f"Log Loss: {log_loss_value:.4f}")

Accuracy: 0.5763
ROC AUC: 0.6034
Log Loss: 0.6751


In [10]:
hero_feature_creator = HeroesFeatureCreator(
    prior_mean=hero_features_hyperparams.prior_mean,
    prior_count=hero_features_hyperparams.prior_count,
    half_life_days=hero_features_hyperparams.half_life_days,
)
team_feature_creator = TeamFeatureCreator(
    prior_mean=team_features_hyperparams.prior_mean,
    prior_count=team_features_hyperparams.prior_count,
    half_life_days=team_features_hyperparams.half_life_days,
)
player_hero_feature_creator = PlayerHeroFeaturesCreator(
    player_prior_count=player_hero_features_hyperparams.player_prior_count,
    player_half_life_days=player_hero_features_hyperparams.player_half_life_days,
    hero_prior_count=player_hero_features_hyperparams.hero_prior_count,
    hero_prior_mean=player_hero_features_hyperparams.hero_prior_mean,
    hero_half_life_days=player_hero_features_hyperparams.hero_half_life_days,
)